# Sanity check: do the trained models use the images? (Kaggle, ~15 min)

Evaluates every trained probe checkpoint it can find under three conditions on held-out windows: correct images, images from a different episode, no images. If the last two are as good as the first, the vision encoder is not doing anything.

**Inputs to attach (Add Input → Your Work):** `so101-extract` (features), `so101-train2` (DINOv2 checkpoints), `so101-pixel-mae` (pixel checkpoints). To test on the separate grasp_1 batch, pin `so101-extract` to **Version 1** in the input's version picker; with the latest version it uses the last 78 episodes of grasp_2 (the validation split).

Accelerator: GPU. Internet: On.

In [ ]:
GIT_REPO = "https://github.com/yashica-patodia/so101-imitation-learning.git"
import os, glob
os.chdir("/kaggle/working")
!rm -rf /kaggle/working/repo && git clone -q {GIT_REPO} /kaggle/working/repo
os.chdir("/kaggle/working/repo")
feats = sorted({os.path.dirname(f) for f in glob.glob("/kaggle/input/**/meta.json", recursive=True) if glob.glob(os.path.dirname(f) + "/episode_*.npz")})
ckpts = sorted(f for f in glob.glob("/kaggle/input/**/best.pt", recursive=True) if os.path.basename(os.path.dirname(f)) in ("dino_policy", "mae_probe", "pixel_probe"))
print("features:", feats); print("checkpoints:", ckpts)
assert feats and ckpts

In [ ]:
for fdir in feats:
    name = os.path.basename(fdir)
    last_n = "--episodes 78" if name == "grasp_2" else ""   # grasp_2: only the validation episodes; grasp_1: all (never trained on)
    print(f"\n================ {name} {last_n} ================")
    C = " ".join(ckpts)
    !python -u -m nano_vla.train.ablate --checkpoint {C} --features {fdir} {last_n} 2>&1 | grep --line-buffered -v Warning